# HMM Regime Hook (MVP)

Train per-symbol HMM regimes on cointegration_data, persist labels, and demonstrate regime-filtered cointegration testing.


In [3]:
from pathlib import Path
import pandas as pd

from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.data_split import split_data_chronologically
from hypothesis_testing.cointegration.hmm_regimes import train_and_persist_labels

# Config (aligned with test framework defaults)
DATA_DIR = Path('/workspace-hku/hku-data/test_data')
TIMEFRAME = '15m'
PRICE_TYPE = 'mark'
BARS_PER_DAY = 96
ARTIFACTS = Path(__file__).parent / 'artifacts'
LABELS_PATH = ARTIFACTS / f"hmm_labels_{TIMEFRAME}_{PRICE_TYPE}.parquet"

# Load and split data
price_data = load_price_data(
    data_dir=DATA_DIR,
    years_back=1.2,
    symbols=None,
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=None,
)
splits = split_data_chronologically(
    price_data,
    cluster_split=(1.2, 0.8),
    cointegration_split=(0.8, 0.2),
    test_split=(0.2, 0.0),
)
cointegration_data = splits['cointegration_data']

# Train HMMs and persist labels
meta = {
    'timeframe': TIMEFRAME,
    'price_type': PRICE_TYPE,
    'train_range': (
        cointegration_data.index.min().isoformat(),
        cointegration_data.index.max().isoformat(),
    ),
}
labels = train_and_persist_labels(cointegration_data, BARS_PER_DAY, LABELS_PATH, meta=meta)
print('Saved labels to:', LABELS_PATH)
print('Labels preview:')
display(labels.head())


ModuleNotFoundError: No module named 'hypothesis_testing'

In [ ]:
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel

# Demonstration: run parallel test with and without regime filtering on a tiny basket set
# (In practice, use real candidate baskets and larger search)

# Build a tiny candidate list from first few symbols if available
symbols = [c.replace('_close', '') for c in cointegration_data.columns[:3]]
candidate_baskets = [symbols[:2]] if len(symbols) >= 2 else []

print('Candidate baskets:', candidate_baskets)

if candidate_baskets:
    # Baseline (no regimes)
    base_results = test_baskets_cointegration_parallel(
        cointegration_data,
        candidate_baskets,
        max_workers=1,
        batch_size=1,
        deduplicate=False,
    )
    print('Baseline cointegrated baskets:', len(base_results))

    # Regime-filtered (keep low-vol state = 0)
    reg_results = test_baskets_cointegration_parallel(
        cointegration_data,
        candidate_baskets,
        regime_labels=labels,
        include_states={0},
        policy='all',
        max_workers=1,
        batch_size=1,
        deduplicate=False,
    )
    print('Regime-filtered cointegrated baskets:', len(reg_results))
